# Playable Replays — LoRA inference

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joyalzzy/playable-replays/blob/ml/player-ai-inference.ipynb)

This notebook loads the LoRA adapter exported by [`player-ai.ipynb`](./player-ai.ipynb), attaches it to its recorded base model, and runs one bounded JSON generation. It does not train, download replay data, expose a public API, or produce the runtime `/v1/actions` contract. The adapter is for offline `next_sampled_packet_prediction` or `state_analysis`, according to its training manifest.

Use only adapter archives and evaluation records you created or trust. Keep credentials, personal data, proprietary replay data, and unlicensed assets out of the notebook.

## 1. Configuration

When this runs in the same Colab runtime as training, the defaults find the exported adapter and `eval.jsonl` automatically. In a fresh runtime, set `USE_BROWSER_UPLOAD=True` and upload `esports-tracker-lora.zip` when prompted.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
ARTIFACT_ROOT = Path("/content/playable-replays-output" if IN_COLAB else "./.local-data/player-ai")
ADAPTER_ARCHIVE = ARTIFACT_ROOT / "esports-tracker-lora.zip"
ADAPTER_DIR = ARTIFACT_ROOT / "esports-tracker-lora"
EVAL_JSONL_PATH = ARTIFACT_ROOT / "eval.jsonl"
USE_BROWSER_UPLOAD = False  # @param {type:"boolean"}
USE_EVAL_EXAMPLE = True  # @param {type:"boolean"}
MAX_SEQ_LENGTH = 2048  # @param {type:"integer"}
MAX_NEW_TOKENS = 1024  # @param {type:"integer"}

if MAX_SEQ_LENGTH < 1 or MAX_NEW_TOKENS < 1:
    raise ValueError("MAX_SEQ_LENGTH and MAX_NEW_TOKENS must be positive")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(json.dumps({"inColab": IN_COLAB, "artifactRoot": str(ARTIFACT_ROOT.resolve())}, indent=2))

## 2. Install the inference dependency

The cell installs Unsloth only when it is absent. Import Unsloth before Transformers, TRL, or PEFT so its model patches are applied consistently.

In [ ]:
import subprocess

if importlib.util.find_spec("unsloth") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "unsloth"])
    print("Installed Unsloth.")
else:
    print("Unsloth is already installed; leaving the runtime environment unchanged.")

## 3. Locate and inspect the adapter

The training export places adapter files at the root of the ZIP. This cell also accepts a ZIP containing one enclosing directory. It rejects unsafe archive paths and requires exactly one `adapter_config.json`.

In [ ]:
import zipfile

archive_path = ADAPTER_ARCHIVE
if USE_BROWSER_UPLOAD:
    if not IN_COLAB:
        raise RuntimeError("Browser upload is available only inside a Colab runtime")
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one LoRA adapter ZIP")
    archive_path = Path(next(iter(uploaded)))

def extract_trusted_adapter(archive: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f"Unsafe path in adapter archive: {member.filename!r}")
        bundle.extractall(destination)

if not (ADAPTER_DIR / "adapter_config.json").exists():
    if not archive_path.exists():
        raise FileNotFoundError(
            f"No adapter directory at {ADAPTER_DIR} and no archive at {archive_path}. "
            "Run the export cell in player-ai.ipynb or enable USE_BROWSER_UPLOAD."
        )
    extract_trusted_adapter(archive_path, ADAPTER_DIR)

adapter_configs = list(ADAPTER_DIR.rglob("adapter_config.json"))
if len(adapter_configs) != 1:
    raise ValueError(f"Expected exactly one adapter_config.json, found {len(adapter_configs)}")
resolved_adapter_dir = adapter_configs[0].parent
adapter_config = json.loads(adapter_configs[0].read_text(encoding="utf-8"))
manifest_path = resolved_adapter_dir / "training-manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else None
print(json.dumps({
    "adapterDir": str(resolved_adapter_dir.resolve()),
    "baseModel": adapter_config.get("base_model_name_or_path"),
    "adapterType": adapter_config.get("peft_type"),
    "manifest": manifest,
}, indent=2, default=str))

## 4. Load the base model and LoRA adapter

A CUDA GPU is required for this notebook's 4-bit loading path. Loading the adapter may download its recorded base model from Hugging Face if the weights are not already cached.

In [ ]:
from unsloth import FastLanguageModel
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Assign a CUDA GPU runtime before loading the 4-bit model")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(resolved_adapter_dir),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print(f"Loaded adapter on {torch.cuda.get_device_name(0)}")

## 5. Select an input

When `eval.jsonl` is available, the first held-out example supplies the exact system and user messages used by training. Otherwise the notebook uses a small structural smoke test. Replace `CUSTOM_STATE_WINDOW` with a real prepared packet window for meaningful inference; the fallback is not an accuracy test.

In [ ]:
PACKET_SYSTEM_PROMPT = (
    "Given a temporal window of compacted, sampled decoded replay packets, predict only the next sampled packet. "
    "Return JSON with task, prediction, evidenceIds, sourceType, and uncertainty. Preserve packet field names and "
    "dataset-native x/z coordinates exactly as provided. Do not add coaching advice, player intent, win probability, "
    "or claims about events beyond the packet target."
)

CUSTOM_STATE_WINDOW = {
    "schemaVersion": "1.0",
    "stateScope": "offline_decoded_replay_packets",
    "task": "next_sampled_packet_prediction",
    "matchId": "structural-inference-smoke-test",
    "snapshots": [
        {
            "tick": 0,
            "state": {
                "task": "next_sampled_packet_prediction",
                "packetIndex": 0,
                "sampleStride": 4,
                "currentPacket": {"packetType": "CreateHero", "payload": {"champion": "chogath", "net_id": 1073741854, "time": 0.0}},
            },
            "metadata": {"task": "next_sampled_packet_prediction", "uncertainty": "synthetic structural smoke test; not replay evidence"},
        },
        {
            "tick": 0,
            "state": {
                "task": "next_sampled_packet_prediction",
                "packetIndex": 4,
                "sampleStride": 4,
                "currentPacket": {"packetType": "DoSetCooldown", "payload": {"cooldown": 0.0, "display_cooldown": -1.0, "net_id": 1073741854, "slot": 3, "time": 0.0}},
            },
            "metadata": {"task": "next_sampled_packet_prediction", "uncertainty": "synthetic structural smoke test; not replay evidence"},
        },
        {
            "tick": 0,
            "state": {
                "task": "next_sampled_packet_prediction",
                "packetIndex": 8,
                "sampleStride": 4,
                "currentPacket": {"packetType": "DoSetCooldown", "payload": {"cooldown": 0.0, "display_cooldown": -1.0, "net_id": 1073741854, "slot": 45, "time": 0.0}},
            },
            "metadata": {"task": "next_sampled_packet_prediction", "uncertainty": "synthetic structural smoke test; not replay evidence"},
        },
    ],
}

expected_output = None
if USE_EVAL_EXAMPLE and EVAL_JSONL_PATH.exists():
    first_line = next((line for line in EVAL_JSONL_PATH.read_text(encoding="utf-8").splitlines() if line.strip()), None)
    if first_line is None:
        raise ValueError(f"Evaluation file is empty: {EVAL_JSONL_PATH}")
    evaluation_example = json.loads(first_line)
    prompt_messages = evaluation_example["messages"][:2]
    expected_output = json.loads(evaluation_example["messages"][2]["content"])
    input_source = str(EVAL_JSONL_PATH)
else:
    prompt_messages = [
        {"role": "system", "content": PACKET_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(CUSTOM_STATE_WINDOW, sort_keys=True, separators=(",", ":"))},
    ]
    input_source = "CUSTOM_STATE_WINDOW structural smoke test"

print(f"Input source: {input_source}")
print(json.dumps(json.loads(prompt_messages[1]["content"]), indent=2)[:5000])

## 6. Generate and validate JSON

The tokenizer returns both token IDs and an attention mask. Passing the complete mapping to `generate` avoids the ambiguous padding warning produced when only `input_ids` are supplied. A larger output allowance prevents long evidence IDs from truncating otherwise valid JSON.

In [ ]:
encoded = tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

with torch.inference_mode():
    generated = model.generate(
        **encoded,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

prompt_length = encoded["input_ids"].shape[-1]
generated_text = tokenizer.decode(generated[0][prompt_length:], skip_special_tokens=True).strip()
print(generated_text)

try:
    prediction = json.loads(generated_text)
except json.JSONDecodeError as error:
    raise ValueError("Model output is not complete JSON; inspect the text above or increase MAX_NEW_TOKENS") from error

task = prediction.get("task")
required_keys = {"task", "evidenceIds", "sourceType", "uncertainty"}
required_keys.add("prediction" if task == "next_sampled_packet_prediction" else "analysis")
missing_keys = sorted(required_keys - set(prediction))
report = {
    "validJson": True,
    "hasRequiredKeys": not missing_keys,
    "missingKeys": missing_keys,
    "output": prediction,
}
if expected_output is not None:
    expected_packet = expected_output.get("prediction", {}).get("nextSampledPacket", {}).get("packetType")
    predicted_packet = prediction.get("prediction", {}).get("nextSampledPacket", {}).get("packetType")
    report["heldOutPacketType"] = {
        "expected": expected_packet,
        "predicted": predicted_packet,
        "exactMatch": predicted_packet == expected_packet,
    }
print(json.dumps(report, indent=2, ensure_ascii=False))

## Next step

Run multiple held-out records and measure exact packet-type accuracy plus field-level precision/recall before deployment. This adapter's output is not compatible with the product's bot-action `/v1/actions` endpoint; expose it as a separate offline packet-prediction service or retrain against the action contract.